# RF Mixer Downconversion

Simulation of a superheterodyne downconversion stage using the `RFMixer` block. A high-frequency RF signal is mixed with a local oscillator (LO) to produce an intermediate frequency (IF) output.

## Mixer Model

The `RFMixer` performs ideal time-domain multiplication:

$$y(t) = G_{\mathrm{conv}} \cdot x_{\mathrm{RF}}(t) \cdot x_{\mathrm{LO}}(t)$$

For sinusoidal inputs at frequencies $f_{\mathrm{RF}}$ and $f_{\mathrm{LO}}$, the output contains sum and difference frequencies:

$$\cos(2\pi f_{\mathrm{RF}} t) \cdot \cos(2\pi f_{\mathrm{LO}} t) = \frac{1}{2}\left[\cos(2\pi(f_{\mathrm{RF}} - f_{\mathrm{LO}})t) + \cos(2\pi(f_{\mathrm{RF}} + f_{\mathrm{LO}})t)\right]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply PathSim docs matplotlib style
plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope, Spectrum
from pathsim.solvers import RKCK54

from pathsim_rf import RFMixer

## System Setup

We set up a downconversion scenario:
- **RF signal** at 1000 Hz
- **LO** at 900 Hz
- **Expected IF** at 100 Hz (difference) and 1900 Hz (sum)

In [ ]:
# Frequencies
f_rf = 1000.0   # RF signal frequency [Hz]
f_lo = 900.0    # Local oscillator frequency [Hz]
f_if = f_rf - f_lo  # Expected IF frequency [Hz]

# Simulation timestep (must resolve highest frequency)
dt = 1 / (20 * (f_rf + f_lo))
duration = 10 / f_if  # Run for 10 IF cycles

# Build blocks
rf_src = Source(func=lambda t: np.sin(2 * np.pi * f_rf * t))
lo_src = Source(func=lambda t: np.sin(2 * np.pi * f_lo * t))
mixer = RFMixer(conversion_gain=0.0)

# Observe time and frequency domain
sco = Scope(labels=['RF', 'LO', 'IF'])
freq = np.linspace(0, 2500, 500)
spc = Spectrum(freq=freq, labels=['IF'])

## Connections

The RF source connects to the mixer's `rf` input (port 0), the LO source to the `lo` input (port 1). The mixer output goes to both a scope and spectrum analyzer.

In [ ]:
connections = [
    Connection(rf_src, mixer["rf"], sco[0]),
    Connection(lo_src, mixer["lo"], sco[1]),
    Connection(mixer, sco[2], spc),
]

sim = Simulation(
    [rf_src, lo_src, mixer, sco, spc],
    connections,
    dt=dt,
    Solver=RKCK54,
    tolerance_lte_rel=1e-5,
    tolerance_lte_abs=1e-7
)

sim.run(duration)

## Time-Domain Results

The mixer output shows the characteristic beat pattern of two closely spaced frequencies.

In [ ]:
t, [rf, lo, if_out] = sco.read()
t = np.array(t) * 1000  # Convert to ms

fig, axes = plt.subplots(3, 1, figsize=(8, 6), dpi=120, sharex=True)

axes[0].plot(t, rf)
axes[0].set_ylabel('RF [V]')
axes[0].set_title(f'RF Signal ({f_rf:.0f} Hz)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, lo)
axes[1].set_ylabel('LO [V]')
axes[1].set_title(f'LO Signal ({f_lo:.0f} Hz)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, if_out)
axes[2].set_ylabel('IF [V]')
axes[2].set_xlabel('Time [ms]')
axes[2].set_title('Mixer Output (IF)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Frequency-Domain Analysis

The spectrum of the mixer output should show two peaks at the sum and difference frequencies: $f_{\mathrm{IF}} = 100$ Hz and $f_{\mathrm{sum}} = 1900$ Hz.

In [ ]:
f, [G_if] = spc.read()

fig, ax = plt.subplots(dpi=120)
ax.plot(f, np.abs(G_if), linewidth=2)
ax.axvline(x=f_if, color='red', linestyle='--', alpha=0.5, label=f'IF = {f_if:.0f} Hz')
ax.axvline(x=f_rf + f_lo, color='orange', linestyle='--', alpha=0.5, label=f'Sum = {f_rf + f_lo:.0f} Hz')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Magnitude')
ax.set_title('Mixer Output Spectrum')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()